In [ ]:
import torchaudio as ta
from chatterbox_infer.mtl_tts_stream import ChatterboxMultilingualTTS
import time

model = ChatterboxMultilingualTTS.from_pretrained(device="cuda")

In [ ]:
import torch
from IPython.display import Audio
import time

texts = [
    "No way! You actually did it?",
    "Stop right there—what do you think you’re doing?",
    "I can’t believe this… it’s unbelievable!",
    "Shhh, be quiet. Did you hear that noise?",
    "Come on, hurry up! We’re going to be late!",
    "Oh please, don’t tell me you forgot again.",
    "Wait, what?! That makes no sense at all.",
    "I told you already—I’m not going back there.",
    "Ha! That was the funniest thing I’ve ever seen.",
    "Ugh, I’m so tired… I just need one more coffee.",
    "Look out! That car is coming way too fast!",
    "Yes, yes, yes! Finally, it worked!",
    "Oh no… I really messed up this time.",
    "Are you kidding me right now?",
    "Listen carefully: this is the most important part.",
]

total_seconds = 0
total_latency = 0

for text in texts:
    st = time.time()
    wavs = []
    last_length = 0
    async for evt in model.generate_stream(
        text,
        audio_prompt_path='./elevenlabs3.mp3',
        language_id='en',
        exaggeration=0.6,
        cfg_weight=0.4,
        temperature=0.7,
        repetition_penalty=1.3,
        min_p=0.02,
        top_p=0.9,
    ):
        if evt["type"] == "chunk":
            wav = evt["audio"]
            wavs.append(wav[:, max(last_length, 0):])
            last_length = wav.shape[-1]
        elif evt["type"] == "eos":
            full = torch.cat(wavs, dim=-1).cpu().numpy()
            # print(f"✅ 스트리밍 끝! {text} ({full.shape[-1]/24000:.2f}s, {time.time()-st:.2f}s 소요)")
            # display(Audio(full, rate=24000))
            total_seconds += full.shape[-1]/24000
            total_latency += time.time() - st

print(f"1초 생성에 {total_latency/total_seconds:.3f}초 걸림")

In [ ]:
import torchaudio as ta
from chatterbox_infer.mtl_tts import ChatterboxMultilingualTTS
import time

model = ChatterboxMultilingualTTS.from_pretrained(device="cuda")

In [ ]:
import torch
from IPython.display import Audio

texts = [
    "No way! You actually did it?",
    "Stop right there—what do you think you’re doing?",
    "I can’t believe this… it’s unbelievable!",
    "Shhh, be quiet. Did you hear that noise?",
    "Come on, hurry up! We’re going to be late!",
    "Oh please, don’t tell me you forgot again.",
    "Wait, what?! That makes no sense at all.",
    "I told you already—I’m not going back there.",
    "Ha! That was the funniest thing I’ve ever seen.",
    "Ugh, I’m so tired… I just need one more coffee.",
    "Look out! That car is coming way too fast!",
    "Yes, yes, yes! Finally, it worked!",
    "Oh no… I really messed up this time.",
    "Are you kidding me right now?",
    "Listen carefully: this is the most important part.",
]

total_seconds = 0
total_latency = 0
for text in texts:
    last_length = 0
    st = time.time()
    wavs = []
    async for evt in model.generate_stream(
            text,
            audio_prompt_path='./elevenlabs3.mp3', 
            language_id='en',
            exaggeration=0.5,
            cfg_weight=0.4,
            temperature=0.7,
            repetition_penalty=1.3,
            min_p=0.02,
            top_p=0.9
        ):
        if evt["type"] == "chunk":
            wav = evt["audio"]
            wavs.append(wav[:, max(last_length, 0):])
            last_length = wav.shape[-1]
        elif evt["type"] == "eos":
            audio = torch.concat(wavs, dim=-1).cpu().numpy()
            # print(f"[For {(audio.shape[-1]-last_length)/24000}s] - taken {time.time() - st:.3f}")
            # display(Audio(audio, rate=24000))   # 예: 오디오 출력 함수
            total_seconds += audio.shape[-1]/24000
            total_latency += time.time() - st

print(f"1초 생성에 {total_latency/total_seconds:.3f}초 걸림")

In [ ]:
overlap = int(24000 * 0.01)

for text in texts:
    last_length, wavs = 0, []
    async for evt in model.generate_stream(
        text,
        audio_prompt_path='./elevenlabs3.mp3',
        language_id='en',
        exaggeration=0.5,
        cfg_weight=0.4,
        temperature=0.7,
        repetition_penalty=1.3,
        min_p=0.02,
        top_p=0.9
    ):
        if evt["type"] == "chunk":
            wav = evt["audio"]
            cut = max(last_length - overlap, 0)
            cur = wav[:, cut:]
            if wavs:
                n = min(overlap, wavs[-1].shape[-1], cur.shape[-1])
                f = torch.linspace(0, 1, n, device=cur.device)
                wavs[-1][:, -n:] = wavs[-1][:, -n:] * (1-f) + cur[:, :n] * f
                cur = cur[:, n:]
            wavs.append(cur)
            last_length = wav.shape[-1]
        elif evt["type"] == "eos":
            full = torch.cat(wavs, -1)
            display(Audio(full.cpu().numpy(), rate=24000))

In [ ]:
# ② 스트리밍 호출(비동기 제너레이터)
async for msg in model.generate_stream(
    text="Hi there, this is a streaming TTS test.",
    language_id="en",
    repetition_penalty=2.0,
    min_p=0.05,
    top_p=1.0,
    exaggeration=0.6,
    cfg_weight=0.5,
    temperature=0.8,
    chunk_size=20,
):
    if msg["type"] == "chunk":
        wav: torch.Tensor = msg["audio"]  # shape: (1, T), sr = S3GEN_SR
        # 여기서 재생/저장/전송 등 원하는 처리
        print("got chunk:", wav.shape)
    elif msg["type"] == "eos":
        print("done")
        break

In [ ]:
from IPython.display import Audio
import time

text = "Where are you?"
out = 'hmhmsas.wav'
tt = 0
for _ in range(2):
    st = time.time()
    wav = model.generate(
        text, 
        audio_prompt_path='./elevenlabs2.mp3', 
        language_id='en',
        exaggeration=0.5,
        cfg_weight=0.4,
        temperature=0.7,
        repetition_penalty=1.3,
        min_p=0.02,
        top_p=0.9,
    )
    print(f"[For {wav.shape[-1]/24000}s] taken: {time.time() - st:.3f}")
    tt += time.time() - st
    display(Audio(wav, rate=24000))
print(f"Average time taken: {tt/10:.3f}")

In [ ]:
from IPython.display import Audio
import torch

display(Audio(wav, rate=24000))
bi=0
all_audios = []
for i in [20, 40, 60, 80, 100, 120, 140, 160]:
    cwav, _ = model.s3gen.inference(
        speech_tokens=tokens[bi:i],
        ref_dict=model.conds.gen,
    )
    all_audios.append(cwav.detach().cpu())
    # display(Audio(cwav.detach().cpu(), rate=24000))
    bi = i

In [ ]:
summed_wav = torch.concat(all_audios, dim=1)
display(Audio(summed_wav, rate=24000))

In [ ]:
wav.shape

In [ ]:
import torchaudio as ta
from chatterbox.tts import ChatterboxTTS
import time
from IPython.display import Audio

model = ChatterboxTTS.from_pretrained(device="cuda")

In [ ]:
text = "Oh, I really love it.. y'know, it's amazing."

last_length = 0
st = time.time()
wavs = []
async for evt in model.generate_stream(text, audio_prompt_path='test-1.wav'):
    if evt["type"] == "chunk":
        wav = evt["audio"]
        wavs.append(wav[:, max(0, last_length-1000):])
        print(f"[{(wav.shape[-1]-last_length)/24000}s] - {time.time() - st}")
        last_length = wav.shape[-1]
        st = time.time()
    elif evt["type"] == "eos":
        print("✅ 스트리밍 끝!")

In [ ]:
import torch, time

SR = 24000
OVERLAP = int(0.05 * SR)  # 50ms

text = "Oh, I really love it.. y'know, it's amazing."

last_length = 0          # 모델이 지금까지 만든 전체 wav 길이
last_tail = None         # 직전 출력 청크의 꼬리(OVERLAP 샘플)
st = time.time()

wavs = []                # overlap-add로 섞은 최종 출력 청크들

async for evt in tts_model.generate_stream(text, audio_prompt_path='test-1.wav'):
    if evt["type"] == "chunk":
        wav = evt["audio"]                    # shape: (ch, T_total_so_far)
        device = wav.device
        dtype = wav.dtype

        # 이번에 "새로 추가된" 구간만 잘라오기
        new_total = wav.shape[-1]
        delta = new_total - last_length
        if delta <= 0:
            continue  # 새로 생긴 게 없으면 스킵

        new_part = wav[:, last_length:new_total]  # (ch, delta)

        if last_tail is None:
            # 첫 청크면 바로 내보냄
            out_chunk = new_part
        else:
            # 겹치는 길이 L (= min(OVERLAP, 새로 생긴 길이, last_tail 길이))
            L = min(OVERLAP, new_part.shape[-1], last_tail.shape[-1])
            if L > 0:
                # last_tail의 마지막 L 샘플 ↔ new_part의 앞 L 샘플을 교차페이드
                fade_in  = torch.linspace(0, 1, L, device=device, dtype=dtype)
                fade_out = 1.0 - fade_in

                mixed = last_tail[:, -L:] * fade_out + new_part[:, :L] * fade_in
                tail  = new_part[:, L:]  # 비겹침 뒷부분

                out_chunk = torch.cat([mixed, tail], dim=-1)
            else:
                # 겹칠 게 없으면 그냥 이어붙임
                out_chunk = new_part

        # 다음 교차페이드를 위해 꼬리 갱신
        # (모델 기준 전체 wav의 최신 꼬리를 쓰는 게 안전)
        new_tail_start = max(0, new_total - OVERLAP)
        last_tail = wav[:, new_tail_start:new_total].detach()

        # 사용자 출력/저장을 위해 overlap-add 결과만 모음
        wavs.append(out_chunk)

        print(f"[{out_chunk.shape[-1]/SR:.3f}s] - {time.time() - st:.3f}")
        last_length = new_total
        st = time.time()

    elif evt["type"] == "eos":
        print("✅ 스트리밍 끝!")

In [ ]:
import torch

display(Audio(wav.cpu().numpy(), rate=24000))   # 예: 오디오 출력 함수
display(Audio(torch.concat(wavs, dim=-1).cpu().numpy(), rate=24000))   # 예: 오디오 출력 함수

In [ ]:
st = time.time()
model.t3.speech_pos_emb.get_fixed_embedding(213)
time.time() - st

st = time.time()
emb = torch.nn.Embedding(113, 512)
emb(torch.tensor(12))
time.time() - st

In [ ]:
st = time.time()
torch.tensor(132).to('cuda')
time.time() - st

In [ ]:
logits = torch.randn((132001))
st = time.time()
probs = torch.softmax(logits, dim=-1)
next_token = torch.multinomial(probs, num_samples=1)  # shape: (B, 1)
print(time.time() - st)

st = time.time()
dist = torch.distributions.Categorical(logits=logits)  # GPU에서 동작
next_token = dist.sample()  # (B, 1) 형태 맞추기
print(time.time() - st)


In [ ]:
import librosa
import time

audio, sr = librosa.load('test-1.wav')

text = "Oh"
st = time.time()
wav, tokens = model.generate(text, audio_prompt_path=audio)
print(wav.shape, tokens.shape, time.time() - st)
# ta.save("test-1.wav", wav, model.sr)
display(Audio('test-1.wav'))

In [ ]:
import torchaudio as ta
from chatterbox_infer.mtl_tts import ChatterboxTTS
import time

model = ChatterboxTTS.from_pretrained(device="cuda")

In [ ]:
from IPython.display import Audio
import librosa
import torch

audio, _ = librosa.load('/workspace/chatterbox/sesame.wav', sr=16000, mono=True)

text = "I want to go home now please let me go."
wav, tokens = model.generate(text, audio_prompt_path=audio)
print(wav.shape[-1]/24000)

out = 'just.wav'
ta.save(out, wav, model.sr)
display(Audio(out))

In [ ]:
display(Audio(audio, rate=16000))

In [ ]:
import torchaudio as ta
from chatterbox_infer.mtl_tts import ChatterboxMultilingualTTS
import time

model = ChatterboxMultilingualTTS.from_pretrained(device="cuda")

In [ ]:
from IPython.display import Audio
import time
import torch

texts = [
    "No way! You actually did it?",
    "Stop right there—what do you think you’re doing?",
    "I can’t believe this… it’s unbelievable!",
    "Shhh, be quiet. Did you hear that noise?",
    "Come on, hurry up! We’re going to be late!",
    "Oh please, don’t tell me you forgot again.",
    "Wait, what?! That makes no sense at all.",
    "I told you already—I’m not going back there.",
    "Ha! That was the funniest thing I’ve ever seen.",
    "Ugh, I’m so tired… I just need one more coffee.",
    "Look out! That car is coming way too fast!",
    "Yes, yes, yes! Finally, it worked!",
    "Oh no… I really messed up this time.",
    "Are you kidding me right now?",
    "Listen carefully: this is the most important part.",
]

total_seconds = 0
total_latency = 0
for text in texts:
    st = time.time()

    wav = model.generate(
        text, 
        audio_prompt_path='./elevenlabs3.mp3', 
        language_id='en',
        exaggeration=0.4,
        cfg_weight=0.0,
        temperature=0.7,
        repetition_penalty=1.3,
        min_p=0.02,
        top_p=0.9,
    )
    total_seconds += wav.shape[-1]/24000
    total_latency += time.time() - st
    print(text)
    # print(f"[For {wav.shape[-1]/24000}s] taken: {time.time() - st:.3f}")
    # display(Audio(wav, rate=24000))

print(f"1초 생성에 {total_latency/total_seconds:.3f}초 걸림")

In [1]:
from IPython.display import Audio
import time
import torch

texts = [
    "No way! You actually did it?",
    "Stop right there—what do you think you’re doing?",
    "I can’t believe this… it’s unbelievable!",
    "Shhh, be quiet. Did you hear that noise?",
    "Come on, hurry up! We’re going to be late!",
    "Oh please, don’t tell me you forgot again.",
    "Wait, what?! That makes no sense at all.",
    "I told you already—I’m not going back there.",
    "Ha! That was the funniest thing I’ve ever seen.",
    "Ugh, I’m so tired… I just need one more coffee.",
    "Look out! That car is coming way too fast!",
    "Yes, yes, yes! Finally, it worked!",
    "Oh no… I really messed up this time.",
    "Are you kidding me right now?",
    "Listen carefully: this is the most important part.",
]

total_seconds = 0
total_latency = 0
for text in texts:
    st = time.time()

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        wav = model.generate(
            text, 
            audio_prompt_path='./elevenlabs3.mp3', 
            language_id='en',
            exaggeration=0.4,
            cfg_weight=0.0,
            temperature=0.7,
            repetition_penalty=1.3,
            min_p=0.02,
            top_p=0.9,
        )
        total_seconds += wav.shape[-1]/24000
        total_latency += time.time() - st
        print(text)
        # print(f"[For {wav.shape[-1]/24000}s] taken: {time.time() - st:.3f}")
        # display(Audio(wav, rate=24000))

print(f"1초 생성에 {total_latency/total_seconds:.3f}초 걸림")

NameError: name 'model' is not defined